# Modeloplossing — Checkpoint Herhalingslabo Classificatie

Dit is de **modeloplossing** van het checkpoint. De bestandsnaam mag niet gewijzigd worden: de checks in `checks/` verwijzen naar dit notebook.

De opgave-versie (`week05/herhaling_classification_checkpoint_opgave.ipynb`) heeft dezelfde celstructuur met TODO's die de student invult. Gebruik overal `test_size=0.2` en `random_state=42`.

Hoe weet je wanneer je welk algoritme best gebruikt ? Gebruik volgende handleiding:

**KNearestNeighbours**
- Geschikt voor classificatie en regressie
- Voor kleine/middelgrote datasets
- heel simpel, kan snel worden getrained
- werkt niet goed voor datasets met heel veel features

**Support Vector Machines**
- Vooral een standaard voor binaire classificatie, of wanneer je een duidelijke marge wilt op classificatie (=het hoofddoel van het algoritme)
- Voor kleine/middelgrote datasets
- Is kostelijker om te trainen dan KNN, maar classificeert dan wel sneller
- Kernel keuze en hyperparameter tuning zijn essentieel

**Decision Trees**
- Wanneer je een 'white box', 'explainable' model wilt
- kan zowel numerieke als categorische features makkelijk aan
- kan heel goed om met missende datapunten
- het risico op overfitting is groot. Bij twijfel, gebruik een random forest

**Random forest**
- Zelfde voordelen als decision tree, minder risico op overfitting
- Geschikt voor middelgrote/grote datasets
- Geschikt voor data met (heel) veel features
- Hebben uiteraard meer trainingstijd nodig dan een enkele decision tree

**Adaboost**
- Als data niet goed is gebalanceerd (niet evenveel items van elke klasse)
- Vermijd een onderfit. Dit model is beter dan 1 enkele onderliggende decision tree

**XGBoost**
- XGBoost is bijna *altijd* een goede keuze
- Middelgrote / grote datasets
- Datasets met (heel) veel features
- De implementatie van XGBoost staat, ondanks dat het een boosting algoritme is, toch deels parallellisatie toe.

In de praktijk zijn er vaak meerdere goede keuzes. Het is belangrijk te weten waarom een bepaald model wel/niet een goede keuze zou zijn, en vervolgens de meest veelbelovende modellen te proberen en te kijken wat het beste werkt.

## Oefening 1 — Diabetes

De dataset `data/diabetes.csv` bevat medische metingen; in de kolom `Outcome` staat of iemand diabetes heeft (1) of niet (0). Dit is een **kleine dataset** (768 rijen, 8 features).

Train twee modellen en vergelijk:
1. Een `SVC` met `kernel='linear'`
2. Een `KNeighborsClassifier` met `n_neighbors=5`

Gebruik `test_size=0.2` en `random_state=42`. Bewaar de voorspellingen in `y_pred_diabetes_svc` / `y_pred_diabetes_knn` en de accuracies in `accuracy_diabetes_svc` / `accuracy_diabetes_knn`.

In [2]:
# Libraries inladen en data klaarzetten
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

diabetes = pd.read_csv('../data/diabetes.csv')

X = diabetes.drop(columns=['Outcome'])
y = diabetes['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# Train een SVC met kernel='linear' op de trainingsdata
model = SVC(kernel='linear', random_state=42)
model.fit(X_train, y_train)

# Maak voorspellingen op de testdata en bewaar ze in y_pred_diabetes_svc
y_pred_diabetes_svc = model.predict(X_test)

# Bereken de accuracy en bewaar ze in accuracy_diabetes_svc
accuracy_diabetes_svc = accuracy_score(y_test, y_pred_diabetes_svc)

# Print de confusion matrix en het classification report ter controle
print(f"Accuracy: {accuracy_diabetes_svc:.2f}")
print(confusion_matrix(y_test, y_pred_diabetes_svc))
print(classification_report(y_test, y_pred_diabetes_svc))

Accuracy: 0.75
[[80 19]
 [19 36]]
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        99
           1       0.65      0.65      0.65        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154



In [4]:
# Train een KNeighborsClassifier met n_neighbors=5 op de trainingsdata
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

# Maak voorspellingen op de testdata en bewaar ze in y_pred_diabetes_knn
y_pred_diabetes_knn = model.predict(X_test)

# Bereken de accuracy en bewaar ze in accuracy_diabetes_knn
accuracy_diabetes_knn = accuracy_score(y_test, y_pred_diabetes_knn)

# Vergelijk met de accuracy van het SVC-model: welk model presteert het beste?
print(f"Accuracy: {accuracy_diabetes_knn:.2f}")

Accuracy: 0.66


## Oefening 2 — Customer Churn

De dataset `data/Telco-Customer-Churn.csv` bevat klantgegevens van een telecombedrijf; de kolom `Churn` geeft aan of een klant afhaakt (`Yes`/`No`). Dit is een **middelgrote dataset** (~7000 rijen) met zowel numerieke als categorische features, dus er is preprocessing nodig.

Zet een pipeline op met een `ColumnTransformer` die:
- numerieke features standaardiseert met een `StandardScaler`
- categorische features one-hot encodeert met een `OneHotEncoder(handle_unknown='ignore')`

Gebruik als classifier een `RandomForestClassifier(n_estimators=100, random_state=42)`. Splits met `test_size=0.2` en `random_state=42`. Bewaar de voorspellingen in `y_pred_churn` en de accuracy in `accuracy_churn`.

In [ ]:
# Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Data inlezen
churn_data = pd.read_csv('../data/Telco-Customer-Churn.csv')

# Features en labels
X = churn_data.drop(columns=['Churn'])
y = churn_data['Churn']

# Splits met test_size=0.2 en random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor: numerieke features standardiseren, categorische features one-hot encoderen
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

# Pipeline: preprocessing + RandomForest, en fitten
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)),
])
model.fit(X_train, y_train)

# Voorspellingen
y_pred_churn = model.predict(X_test)

# Evaluatie
accuracy_churn = accuracy_score(y_test, y_pred_churn)
print(f"Accuracy: {accuracy_churn:.2f}")
print(confusion_matrix(y_test, y_pred_churn))
print(classification_report(y_test, y_pred_churn))

## Oefening 3 — Creditcardfraude

De dataset `data/creditcard.csv` bevat ~285.000 transacties waarvan de features al gepreprocessesd zijn (kolommen `V1` t/m `V28`, `Time`, `Amount`). De kolom `Class` is 1 bij fraude en 0 bij een valide transactie. Dit is een **grote dataset** én **sterk ongebalanceerd** (~500 fraudegevallen).

Train een `AdaBoostClassifier(n_estimators=50, random_state=42)` en een `XGBClassifier(n_estimators=50, random_state=42)` en vergelijk beide. Splits met `test_size=0.2` en `random_state=42`.

Bewaar de voorspellingen in `y_pred_creditcard_ada` / `y_pred_creditcard_xgb` en de accuracies in `accuracy_creditcard_ada` / `accuracy_creditcard_xgb`. Kijk zeker naar de confusion matrix / classification report, niet alleen naar de accuracy.

In [ ]:
# Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Dataset inlezen
creditcard_data = pd.read_csv('../data/creditcard.csv')

# Features en labels
X = creditcard_data.drop(columns=['Class'])
y = creditcard_data['Class']

# Splits met test_size=0.2 en random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# AdaBoost
ada_model = AdaBoostClassifier(n_estimators=50, random_state=42)
ada_model.fit(X_train, y_train)
y_pred_creditcard_ada = ada_model.predict(X_test)
accuracy_creditcard_ada = accuracy_score(y_test, y_pred_creditcard_ada)
print("AdaBoost Results:")
print(f"Accuracy: {accuracy_creditcard_ada:.4f}")
print(confusion_matrix(y_test, y_pred_creditcard_ada))
print(classification_report(y_test, y_pred_creditcard_ada))

# XGBoost
xgb_model = XGBClassifier(n_estimators=50, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_creditcard_xgb = xgb_model.predict(X_test)
accuracy_creditcard_xgb = accuracy_score(y_test, y_pred_creditcard_xgb)
print("XGBoost Results:")
print(f"Accuracy: {accuracy_creditcard_xgb:.4f}")
print(confusion_matrix(y_test, y_pred_creditcard_xgb))
print(classification_report(y_test, y_pred_creditcard_xgb))

## Meerkeuzevragen

### Vraag 1
Je hebt een dataset met 500 rijen en 20 features. De business wil graag begrijpen **welke** factoren tot de voorspelling leiden. Welk model kies je bij voorkeur?

- **A)** Random Forest met 500 bomen
- **B)** Decision Tree
- **C)** XGBoost met 500 bomen
- **D)** Support Vector Machine

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_algoritme`.

### Vraag 2
De creditcard-dataset is sterk ongebalanceerd. Je model haalt 99.8% accuracy. Wat is de beste manier om te beoordelen of het model bruikbaar is voor het detecteren van fraude?

- **A)** Accuracy is het hoogst mogelijke getal, dus het model is uitstekend
- **B)** Kijk naar de confusion matrix / classification report, en vooral naar de detectie van de fraudeklasse
- **C)** Mean Squared Error berekenen
- **D)** R²-score berekenen

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_ongebalanceerd`.

In [ ]:
# Vraag 1: een decision tree is interpreteerbaar ('white box') en past bij 500 rijen;
# ensembles met 500 bomen overfitten op 500 datapunten.
antwoord_algoritme = "B"

# Vraag 2: bij sterk ongebalanceerde data zegt accuracy weinig;
# de confusion matrix / classification report toont hoe goed de fraudeklasse gedetecteerd wordt.
antwoord_ongebalanceerd = "B"